# No Discussion Voting Mad

In [1]:
import polars as pl

import src.social_groups.polars_columns as plc
from social_groups.analysis.polars_transformations.apply_parsing_and_group_decision import (
    apply_parsing_and_group_decision,
)
from social_groups.polars_values import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)

In [2]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [3]:
from social_groups.analysis.definitions import defs

frame = defs().load_asset_value("no_discussion_voting")

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:111: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-03-09 11:37:22 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/no_discussion_voting.parquet using PolarsParquetIOManager...


In [4]:
frame.head()

shape: (5, 18)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ question   ┆ answer_str ┆ model_name ┆ temperatur │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ ---        ┆ ing        ┆ s          ┆ e          │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ str        ┆ ---        ┆ ---        ┆ ---        │
│      ┆        ┆             ┆ str        ┆   ┆            ┆ str        ┆ list[str]  ┆ str        │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 9820 ┆ 112    ┆ 25          ┆ 4e5dfcf095 ┆ … ┆ Q: What    ┆ J          ┆ ["Qwen/Qwe ┆ 0.1        │
│      ┆        ┆             ┆ fcf284     ┆   ┆ stable     ┆            ┆ n3-4B"]    ┆            │
│      ┆        ┆             ┆            ┆   ┆ isotope is ┆            ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆ comm…      ┆            ┆            ┆            │
│ 9821 ┆ 112    ┆ 0           ┆ cab6934a79 ┆ … ┆ Q: What    ┆ B          ┆ ["Qwen/Qwe ┆ 0.1        │
│      ┆        ┆             ┆ c4a106     ┆   ┆ will       ┆            ┆ n3-4B"]    ┆            │
│      ┆        ┆             ┆            ┆   ┆ happen to  ┆            ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆ the equ…   ┆            ┆            ┆            │
│ 9822 ┆ 112    ┆ 26          ┆ e7d5a2b2bb ┆ … ┆ Q: An      ┆ I          ┆ ["Qwen/Qwe ┆ 0.1        │
│      ┆        ┆             ┆ ad97fe     ┆   ┆ electric   ┆            ┆ n3-4B"]    ┆            │
│      ┆        ┆             ┆            ┆   ┆ dipole     ┆            ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆ consisti…  ┆            ┆            ┆            │
│ 9823 ┆ 112    ┆ 32          ┆ 4c5e046b14 ┆ … ┆ Q: In      ┆ C          ┆ ["Qwen/Qwe ┆ 0.1        │
│      ┆        ┆             ┆ 42d2a6     ┆   ┆ building a ┆            ┆ n3-4B"]    ┆            │
│      ┆        ┆             ┆            ┆   ┆ linear     ┆            ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆ regres…    ┆            ┆            ┆            │
│ 9824 ┆ 112    ┆ 31          ┆ 2d6d40309f ┆ … ┆ Q:  When   ┆ A          ┆ ["Qwen/Qwe ┆ 0.1        │
│      ┆        ┆             ┆ a984ce     ┆   ┆ was the    ┆            ┆ n3-4B"]    ┆            │
│      ┆        ┆             ┆            ┆   ┆ major      ┆            ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆ shift b…   ┆            ┆            ┆            │
└──────┴────────┴─────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

In [5]:
data = apply_parsing_and_group_decision(
    frame.with_columns(
        model_name=pl.col(plc.model_names)
        .list.item()
        .replace(MODEL_NAME_TO_LETTER_MAPPING)
    ).drop(plc.model_names),
    parser,
    comparer,
    group_reply,
)

In [6]:
data.group_by("temperature", "model_name").agg(
    pl.col(plc.is_correct).mean().alias(plc.accuracy)
)

temperature,model_name,accuracy
str,str,f64
"""0.0""","""L""",0.32
"""0.0""","""H""",0.75
"""0.3""","""L""",0.35
"""0.3""","""M""",0.6
"""0.1""","""H""",0.75
"""0.0""","""M""",0.62
"""0.1""","""L""",0.4
"""0.1""","""M""",0.64
"""0.3""","""H""",0.74


#### Does someone know the correct answer?

In [7]:
data.head()

shape: (5, 23)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ ___parsed_ ┆ ___parsed_ ┆ ___parsed_ ┆ is_correct │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ individual ┆ combined_a ┆ combined_a ┆ ---        │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ _answers_a ┆ nswers_bef ┆ nswers_aft ┆ bool       │
│      ┆        ┆             ┆ str        ┆   ┆ …          ┆ …          ┆ …          ┆            │
│      ┆        ┆             ┆            ┆   ┆ ---        ┆ ---        ┆ ---        ┆            │
│      ┆        ┆             ┆            ┆   ┆ list[str]  ┆ str        ┆ str        ┆            │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 9820 ┆ 112    ┆ 25          ┆ 4e5dfcf095 ┆ … ┆ ["J", "J", ┆ J          ┆ J          ┆ true       │
│      ┆        ┆             ┆ fcf284     ┆   ┆ … "J"]     ┆            ┆            ┆            │
│ 9821 ┆ 112    ┆ 0           ┆ cab6934a79 ┆ … ┆ ["B", "B", ┆ B          ┆ B          ┆ true       │
│      ┆        ┆             ┆ c4a106     ┆   ┆ … "B"]     ┆            ┆            ┆            │
│ 9822 ┆ 112    ┆ 26          ┆ e7d5a2b2bb ┆ … ┆ ["I", "I", ┆ I          ┆ I          ┆ true       │
│      ┆        ┆             ┆ ad97fe     ┆   ┆ … "I"]     ┆            ┆            ┆            │
│ 9823 ┆ 112    ┆ 32          ┆ 4c5e046b14 ┆ … ┆ ["D", "D", ┆ D          ┆ D          ┆ false      │
│      ┆        ┆             ┆ 42d2a6     ┆   ┆ … "D"]     ┆            ┆            ┆            │
│ 9824 ┆ 112    ┆ 31          ┆ 2d6d40309f ┆ … ┆ ["C", "C", ┆ C          ┆ C          ┆ false      │
│      ┆        ┆             ┆ a984ce     ┆   ┆ … "C"]     ┆            ┆            ┆            │
└──────┴────────┴─────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

In [8]:
from social_groups.analysis.polars_transformations.apply_comparer_to_list_column import (
    apply_comparer_to_list_column,
)

ind_data = apply_comparer_to_list_column(
    data.with_columns(
        is_correct_individual=pl.col(plc.parsed_individual_answers_after)
    ),
    comparer,
    "is_correct_individual",
)

In [15]:
ind_data.schema

Schema([('id', Int64),
        ('run_id', Int64),
        ('question_id', Int64),
        ('phoenix_span_id', String),
        ('phoenix_span_url', String),
        ('run_identifier', String),
        ('answers_at_beginning', List(String)),
        ('answers_at_end', List(String)),
        ('experiment_id', Int64),
        ('experiment_configuration_json', String),
        ('meta_info_json', String),
        ('name', String),
        ('original_question_id', Int64),
        ('category', String),
        ('question', String),
        ('answer_string', String),
        ('temperature', String),
        ('model_name', String),
        ('___parsed_individual_answers_before___', List(String)),
        ('___parsed_individual_answers_after___', List(String)),
        ('___parsed_combined_answers_before___', String),
        ('___parsed_combined_answers_after___', String),
        ('is_correct', Boolean),
        ('is_correct_individual', List(Boolean))])

In [9]:
plot_hist_data = (
    ind_data.with_columns(
        num_correct_individually=pl.col("is_correct_individual").list.count_matches(
            True
        )
    ).select("temperature", "model_name", "num_correct_individually")
    # .len("occurrences").sort("temperature", "model_name", "num_correct_individually")
)

In [10]:
plot_hist_data.head()

temperature,model_name,num_correct_individually
str,str,u32
"""0.1""","""M""",12
"""0.1""","""M""",12
"""0.1""","""M""",12
"""0.1""","""M""",0
"""0.1""","""M""",0


In [11]:
import plotly.express as px
import polars as pl

fig = px.histogram(
    plot_hist_data.to_pandas()
    .sort_values("temperature")
    .sort_values("model_name", key=lambda x: x.replace({"L": "1", "M": "2", "H": "3"})),
    x="num_correct_individually",
    facet_col="temperature",
    facet_row="model_name",
    nbins=len(plot_hist_data.to_pandas()["num_correct_individually"].unique()),
    title="Distribution of num_correct by Name and Height",
)

fig.show()

### What influence does number of participants have?

In [12]:
participants_frame = frame.with_columns(
    model_name=pl.col(plc.model_names).list.item().replace(MODEL_NAME_TO_LETTER_MAPPING)
).drop(plc.model_names)

results = None

for i in range(
    1, participants_frame[plc.answers_at_end].list.len().unique().item() + 1
):
    new = (
        apply_parsing_and_group_decision(
            participants_frame.with_columns(
                pl.col(plc.answers_at_end).list.slice(0, i)
            ),
            parser,
            comparer,
            group_reply,
        )
        .group_by("temperature", "model_name")
        .agg(pl.col(plc.is_correct).mean().alias(plc.accuracy))
        .with_columns(participants=pl.lit(i))
    )
    if results is None:
        results = new
    else:
        results = pl.concat([results, new], how="align")

results.head()

temperature,model_name,accuracy,participants
str,str,f64,i32
"""0.0""","""H""",0.67,1
"""0.0""","""H""",0.67,2
"""0.0""","""H""",0.7,3
"""0.0""","""H""",0.71,4
"""0.0""","""H""",0.71,6


In [13]:
print("Spearman Rank Correlation")
for model in results["model_name"].unique():
    print(model)
    for method in {"spearman", "pearson"}:
        print(method)
        print(
            results.sort("accuracy", "participants", descending=True)
            .filter(model_name=model)
            .select(pl.corr("accuracy", "participants", method=method))
            .item()
        )
    print("__________________________________________________________")

Spearman Rank Correlation
M
pearson
0.5404777246499622
spearman
0.5230789615428499
__________________________________________________________
H
pearson
0.8070490378300924
spearman
0.8290165815108214
__________________________________________________________
L
pearson
0.6165896294725991
spearman
0.6195767431081373
__________________________________________________________


In [14]:
results.sort("accuracy", "participants", descending=True).plot.line(
    x="participants", y="accuracy", color="model_name", strokeDash="temperature"
).properties(
    width=400,
    height=300,
)

alt.Chart(...)